In [2]:
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# 1. CONFIGURACIÓN DE CREDENCIALES
# Recuerda: el archivo .json debe estar en tu carpeta y en el .gitignore
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
creds_file = 'tensile-verve-492701-h4-5e040553396f.json' # <--- ASEGÚRATE QUE ESTE NOMBRE SEA EL CORRECTO
creds = ServiceAccountCredentials.from_json_keyfile_name(creds_file, scope)
client = gspread.authorize(creds)

# 2. APERTURA DEL DOCUMENTO
try:
    doc = client.open("Registro de antibióticos (respuestas)")
    print("✅ Conexión establecida con Google Sheets")
    
    # 3. FUNCIÓN DE CARGA SEGURA (Evita el error de duplicados '')
    def cargar_hoja_segura(nombre_hoja):
        hoja = doc.worksheet(nombre_hoja)
        # Traemos todos los valores
        raw_data = hoja.get_all_values()
        # Creamos el DataFrame usando la primera fila como encabezado
        df = pd.DataFrame(raw_data[1:], columns=raw_data[0])
        # Eliminamos columnas fantasma (sin nombre)
        df = df.loc[:, df.columns != '']
        # Limpieza básica de nombres de columnas (quitar espacios)
        df.columns = [c.strip() for c in df.columns]
        return df

    # 4. CARGA DE LAS FUENTES DE DATOS
    df_stock = cargar_hoja_segura("StockAntibióticos")
    df_vencimientos = cargar_hoja_segura("Calc_Vencimientos")
    
    print(f"✅ Hoja Stock: {df_stock.shape[0]} registros cargados.")
    print(f"✅ Hoja Vencimientos: {df_vencimientos.shape[0]} registros cargados.")

    # 5. INTEGRACIÓN (EL "CORAZÓN" DEL PROYECTO)
    # Unimos la información de stock con sus fechas de vencimiento
    # Usamos 'inner' para asegurar que solo vemos lo que coincide en ambas
    df_unificado = pd.merge(
        df_stock, 
        df_vencimientos, 
        left_on='Antibiótico', 
        right_on='Producto', 
        how='inner'
    )
    
    print("\n--- VISTA PREVIA DEL INVENTARIO UNIFICADO ---")
    display(df_unificado.head())

except Exception as e:
    print(f"❌ Error durante el proceso: {e}")

✅ Conexión establecida con Google Sheets
✅ Hoja Stock: 176 registros cargados.
✅ Hoja Vencimientos: 92 registros cargados.

--- VISTA PREVIA DEL INVENTARIO UNIFICADO ---


,Marca temporal,Tipo de registro,Antibiótico,Responsable (iniciales),Fecha de recepción,"Cantidad (En caso de sensidiscos, indicar cantidad de tubos)",Fecha de apertura,Fecha de cierre,Motivo de cierre,Comentario (opcional),Fecha de vencimiento,Cantidad eliminada,Producto,Fecha_Venc,Cantidad_Recibida,ID_Fila,Posición acumulada,Estado lote
0,6/01/2026 8:02:24,Apertura,Aztreonam (Sensidisco),SCS,,,29/08/2025,,,,,,Aztreonam (Sensidisco),21/08/2026,2,38,2,Consumido
1,6/01/2026 8:02:24,Apertura,Aztreonam (Sensidisco),SCS,,,29/08/2025,,,,,,Aztreonam (Sensidisco),19/09/2026,5,39,7,En Reserva
2,6/01/2026 8:02:39,Apertura,Amikacina (Sensidisco),SCS,,,29/08/2025,,,,,,Amikacina (Sensidisco),3/08/2026,2,31,2,Consumido
3,6/01/2026 8:02:39,Apertura,Amikacina (Sensidisco),SCS,,,29/08/2025,,,,,,Amikacina (Sensidisco),30/09/2026,3,32,5,En Uso
4,6/01/2026 8:02:39,Apertura,Amikacina (Sensidisco),SCS,,,29/08/2025,,,,,,Amikacina (Sensidisco),24/01/2028,5,33,10,En Reserva


In [6]:
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# 1. CREDENCIALES
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
creds_file = 'tensile-verve-492701-h4-5e040553396f.json' 
creds = ServiceAccountCredentials.from_json_keyfile_name(creds_file, scope)
client = gspread.authorize(creds)

# 2. FUNCIÓN DE CARGA SEGURA
def extraer_datos_limpios(hoja):
    # Traemos todo el contenido de la hoja
    datos_brutos = hoja.get_all_values()
    # Creamos el DataFrame
    df = pd.DataFrame(datos_brutos[1:], columns=datos_brutos[0])
    # Limpieza: quitamos columnas sin nombre y quitamos espacios en los encabezados
    df = df.loc[:, df.columns != '']
    df.columns = [c.strip() for c in df.columns]
    # Quitamos filas que estén totalmente vacías (limpieza de integridad)
    df = df.dropna(how='all')
    return df

try:
    doc = client.open("Registro de antibióticos (respuestas)")
    
    # 3. EXTRACCIÓN POR ÍNDICE (Garantiza usar solo las 2 primeras hojas)
    print(f"Leyendo documento: {doc.title}")
    
    # Hoja 0: StockAntibióticos / Hoja 1: Calc_Vencimientos
    hoja_1 = doc.get_worksheet(0)
    hoja_2 = doc.get_worksheet(1)
    
    df_stock = extraer_datos_limpios(hoja_1)
    df_venc = extraer_datos_limpios(hoja_2)
    
    print(f"✅ Hoja '{hoja_1.title}' cargada con {len(df_stock)} filas.")
    print(f"✅ Hoja '{hoja_2.title}' cargada con {len(df_venc)} filas.")

    # 4. UNIFICACIÓN INTELIGENTE
    # Usamos 'left' para mantener todo lo de la hoja de Stock 
    # y traer el vencimiento si existe.
    # Nota: Asegúrate que en la Hoja 1 la columna sea 'Antibiótico' 
    # y en la Hoja 2 sea 'Producto'
    df_unificado = pd.merge(
        df_stock, 
        df_venc, 
        left_on='Antibiótico', 
        right_on='Insumo', 
        how='left'
    )

    # 5. LIMPIEZA POST-UNIÓN (Solo borra si existe para evitar el error 'Producto')
    columnas_a_borrar = ['Insumo']
    for col in columnas_a_borrar:
        if col in df_unificado.columns:
            df_unificado = df_unificado.drop(columns=[col])

    print("\n📊 Vista previa del Inventario Unificado (Módulo 2):")
    display(df_unificado.head())

except Exception as e:
    print(f"❌ Error detectado: {e}")
    print("\n💡 Tip del Tutor: Verifica que la Hoja 1 tenga una columna llamada 'Antibiótico' y la Hoja 2 una llamada 'Producto'.")

Leyendo documento: Registro de antibióticos (respuestas)
✅ Hoja 'StockAntibióticos' cargada con 176 filas.
✅ Hoja 'StockInsumos' cargada con 12 filas.

📊 Vista previa del Inventario Unificado (Módulo 2):


,Marca temporal_x,Tipo de registro_x,Antibiótico,Responsable (iniciales),Fecha de recepción_x,"Cantidad (En caso de sensidiscos, indicar cantidad de tubos)",Fecha de apertura_x,Fecha de cierre,Motivo de cierre,Comentario (opcional),Fecha de vencimiento_x,Cantidad eliminada,Marca temporal_y,Tipo de registro_y,Responsable,Fecha de recepción_y,Cantidad,Fecha de apertura_y,Fecha de vencimiento_y
0,6/01/2026 8:02:24,Apertura,Aztreonam (Sensidisco),SCS,,,29/08/2025,,,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6/01/2026 8:02:39,Apertura,Amikacina (Sensidisco),SCS,,,29/08/2025,,,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6/01/2026 8:02:56,Apertura,Ciprofloxacino (Sensidisco),SCS,,,9/12/2025,,,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6/01/2026 8:03:19,Apertura,Cefotaxima (Sensidisco),SCS,,,4/12/2025,,,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6/01/2026 8:05:10,Apertura,Cefoxitin (Sensidisco),SCS,,,4/12/2025,,,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
